In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering

# Load the dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00488/Live_20210128.csv"
df = pd.read_csv(url)

print("Dataset loaded successfully.\n")

# Q1: What is the shape of the data?
initial_shape = df.shape
print(f"Q1: The initial shape of the data is: {initial_shape}")

# Check if any feature contains Null values.
null_counts = df.isnull().sum()
features_with_nulls = null_counts[null_counts > 0]

# Q2: How many features contain Null values?
num_features_with_nulls = len(features_with_nulls)
print(f"Q2: Number of features containing Null values: {num_features_with_nulls}")

# Q3: Column 'status_type' has _ _ _ _ (number) unique values.
unique_status_types = df['status_type'].nunique()
print(f"Q3: Column 'status_type' has {unique_status_types} unique values.\n")

# Drop all the features which have Null values.
df_cleaned = df.drop(columns=features_with_nulls.index)
print(f"Dropped features with null values. New shape: {df_cleaned.shape}\n")

# Save 'status_type' as target variable.
target = df_cleaned['status_type']

# Drop the feature "status_id", "status_type" and "status_published" from training set.
features = df_cleaned.drop(columns=["status_id", "status_type", "status_published"], errors='ignore')

print(f"Features for training (after dropping specified columns): {features.shape}\n")

# Use LabelEncoder to transform the target variable.
le = LabelEncoder()
target_encoded = le.fit_transform(target)
print("Target variable 'status_type' encoded using LabelEncoder.\n")

# Use standard scaler to scale the features.
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)
features_scaled_df = pd.DataFrame(features_scaled, columns=features.columns)
print("Features scaled using StandardScaler.\n")

# Display the first few rows of the scaled features to confirm
print("First 5 rows of scaled features:\n", features_scaled_df.head())


# The next steps will involve K-means and Agglomerative Clustering.
# (Q4-Q8 will be addressed in subsequent steps based on these clustering results.)

Dataset loaded successfully.

Q1: The initial shape of the data is: (7050, 16)
Q2: Number of features containing Null values: 4
Q3: Column 'status_type' has 4 unique values.

Dropped features with null values. New shape: (7050, 12)

Features for training (after dropping specified columns): (7050, 9)

Target variable 'status_type' encoded using LabelEncoder.

Features scaled using StandardScaler.

First 5 rows of scaled features:
    num_reactions  num_comments  num_shares  num_likes  num_loves  num_wows  \
0       0.646104      0.323350    1.686879   0.482727   1.983266  0.196196   
1      -0.173192     -0.252206   -0.304144  -0.144720  -0.318454 -0.147879   
2      -0.006738      0.013089    0.129017  -0.024571   0.206938 -0.033187   
3      -0.257499     -0.252206   -0.304144  -0.231495  -0.318454 -0.147879   
4      -0.037003     -0.252206   -0.304144  -0.024571  -0.093286 -0.147879   

   num_hahas  num_sads  num_angrys  
0   0.076713  0.473570   -0.155748  
1  -0.176010 -0.152587 

In [ ]:
# Q4-Q6: K-means Clustering

# Q4: Train K-means model with k=2 and get inertia score
kmeans_k2 = KMeans(n_clusters=2, random_state=10, n_init=10)
kmeans_k2.fit(features_scaled)
inertia_k2 = kmeans_k2.inertia_
print(f"Q4: Inertia score at k=2: {inertia_k2:.2f}\n")

# Q5: Train K-means model with k=4
kmeans_k4 = KMeans(n_clusters=4, random_state=10, n_init=10)
kmeans_k4_labels = kmeans_k4.fit_predict(features_scaled)

# To answer Q5 and Q6, we need to evaluate against the true labels (target_encoded).
# Since cluster labels are arbitrary, we need to map them to the true labels for 'accuracy'.
# This is a common challenge in clustering evaluation when true labels are known.
# For simplicity, we will find the best mapping between cluster labels and true labels.

from sklearn.metrics import accuracy_score
from scipy.optimize import linear_sum_assignment
import numpy as np

def cluster_accuracy(true_labels, cluster_labels):
    # Ensure labels are numpy arrays
    true_labels = np.array(true_labels)
    cluster_labels = np.array(cluster_labels)

    # Get unique cluster and true labels
    unique_cluster_labels = np.unique(cluster_labels)
    unique_true_labels = np.unique(true_labels)

    # Initialize cost matrix
    # Rows represent cluster labels, columns represent true labels
    cost_matrix = np.zeros((len(unique_cluster_labels), len(unique_true_labels)))

    # Populate cost matrix: cost is the number of disagreements
    for i, cluster_l in enumerate(unique_cluster_labels):
        for j, true_l in enumerate(unique_true_labels):
            # Count how many points in cluster_l are NOT true_l
            cost_matrix[i, j] = np.sum((cluster_labels == cluster_l) & (true_labels != true_l))

    # Solve the minimum cost assignment problem
    row_ind, col_ind = linear_sum_assignment(cost_matrix)

    # Calculate total disagreements
    total_disagreements = cost_matrix[row_ind, col_ind].sum()

    # Calculate accuracy
    accuracy = 1 - (total_disagreements / len(true_labels))
    return accuracy, unique_cluster_labels[row_ind], unique_true_labels[col_ind]

# Calculate accuracy for k=4 K-means
accuracy_k4_kmeans, assigned_cluster_labels_kmeans, assigned_true_labels_kmeans = cluster_accuracy(target_encoded, kmeans_k4_labels)

# Q5: How many labels were predicted accurately at k=4?
# This question can be interpreted as the count of accurately classified samples.
predicted_accurate_count = int(accuracy_k4_kmeans * len(target_encoded))
print(f"Q5: Number of labels predicted accurately at k=4 (K-means): {predicted_accurate_count}\n")

# Q6: What is the label predicted for the first row of samples?
# This refers to the cluster label assigned by K-means at k=4 for the first sample.
first_row_predicted_label_kmeans = kmeans_k4_labels[0]
print(f"Q6: K-means (k=4) label predicted for the first row of samples: {first_row_predicted_label_kmeans}\n")


# Q7-Q8: Agglomerative Clustering

# Train Agglomerative Clustering model
agglomerative = AgglomerativeClustering(n_clusters=4, linkage='ward') # Removed affinity='euclidean'
agglomerative_labels = agglomerative.fit_predict(features_scaled)

# Q7: Enter the number of leaves in the hierarchical tree.
# For AgglomerativeClustering, the number of leaves is simply the number of samples.
# If `n_clusters` is not None, it prunes the tree to form `n_clusters`.
# The 'number of leaves' typically refers to the number of original data points.
# The documentation for AgglomerativeClustering doesn't directly expose 'n_leaves'
# like some other hierarchical clustering methods that build a full dendrogram first.
# However, in the context of a dendrogram, each data point is a leaf before merging.
# So, the number of leaves is the number of samples.
num_leaves = features_scaled.shape[0]
print(f"Q7: Number of leaves in the hierarchical tree (Agglomerative Clustering): {num_leaves}\n")

# Q8: What is the accuracy of the model (in Percentage)?
# Calculate accuracy for Agglomerative Clustering
accuracy_agglomerative, assigned_cluster_labels_agg, assigned_true_labels_agg = cluster_accuracy(target_encoded, agglomerative_labels)
accuracy_percentage_agglomerative = accuracy_agglomerative * 100
print(f"Q8: Accuracy of the Agglomerative Clustering model: {accuracy_percentage_agglomerative:.2f}%\n")

Q4: Inertia score at k=2: 48278.10

Q5: Number of labels predicted accurately at k=4 (K-means): 4477

Q6: K-means (k=4) label predicted for the first row of samples: 0

Q7: Number of leaves in the hierarchical tree (Agglomerative Clustering): 7050

Q8: Accuracy of the Agglomerative Clustering model: 62.58%

